In [1]:
#Import Pandas

import pandas as pd

In [3]:
#Load data sets

cd = pd.read_csv("Customer_Demographics_Cleaned.csv")
ct = pd.read_csv("Customer_Transactions_Cleaned.csv")
smi = pd.read_csv("Social Media Interactions Cleaned.csv")

## 1. CustomerID comparison across Datasets

In [4]:
#Checking CustomerID dtype

print("Customer Demographics datatype:", cd["CustomerID"].dtype)
print("Customer Transactions datatype:", ct["CustomerID"].dtype)
print("Social Media Interactions datatype:", smi["CustomerID"].dtype)

Customer Demographics datatype: str
Customer Transactions datatype: str
Social Media Interactions datatype: str


In [23]:
#Creating Variables for customer per dataset accounding to their CustomerID

cd_customers = set(cd["CustomerID"].dropna())
ct_customers = set(ct["CustomerID"].dropna())
smi_customers = set(smi["CustomerID"].dropna())

#Printing number of unique customers

print("Unique customers:")
print("Demographics:", len(cd_customers))
print("Transactions:", len(ct_customers))
print("Social Media:", len(smi_customers))

#Printing values of customer overlap 

print("\nCustomer overlap:")
print("Demographics + Transactions:", len(cd_customers & ct_customers))
print("Demographics + Social Media:", len(cd_customers & smi_customers))
print("Transactions + Social Media:", len(ct_customers & smi_customers))
print("All three datasets:", len(cd_customers & ct_customers & smi_customers))

Unique customers:
Demographics: 2985
Transactions: 1858
Social Media: 1893

Customer overlap:
Demographics + Transactions: 1848
Demographics + Social Media: 1881
Transactions + Social Media: 1175
All three datasets: 1167


## 2. Age group comparison with the Amount Transaction

In [19]:
#Creating a variable for Customer Age in relate to specific customer ID

cd_age = cd[["CustomerID", "Age"]].dropna()

#Creating a an age bin for different age groups

age_groups = pd.cut(cd_age["Age"],
    bins=[17, 25, 35, 45, 55, 65, 70],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-70"]
)

#Creating a variable for age group

cd_age = cd_age.copy()
cd_age["AgeGroup"] = age_groups

#mergining cd_age on CustomerID

age_amount = ct.merge(
    cd_age[["CustomerID", "AgeGroup"]],
    on="CustomerID",
    how="inner"
)

#Removing data with amount equal to 0 and -100

age_amount = age_amount[
    (age_amount["Amount"] != 0) &
    (age_amount["Amount"] != -100)
]

#Aggregating and printing count mean and median of amount across the age groups

print(
    age_amount.groupby("AgeGroup", observed=True)["Amount"]
    .agg(["count", "mean", "median"])
    .round(2)
)

          count    mean  median
AgeGroup                       
18-25       376  485.00  494.09
26-35       434  486.10  491.62
36-45       834  510.16  497.20
46-55       484  508.38  508.04
56-65       521  505.52  494.09
66-70       220  525.99  508.04


## 3. Amount comparison Across product categories

In [25]:
#Creating a variable for category amount that isnt 0 and -100

category_amount = ct[
    (ct["Amount"] != 0) &
    (ct["Amount"] != -100)
]


#Aggregating and printing count mean median min max amount per product category
print(
    category_amount.groupby("ProductCategory")["Amount"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

                 count    mean  median    min     max
ProductCategory                                      
Automotive         526  507.85  508.04  10.14  999.86
Clothing           551  505.21  491.62  10.21  998.83
Electronics        545  497.66  494.09  10.03  999.86
Health & Beauty    510  483.98  458.76  10.18  999.56
Home & Garden      504  531.33  511.64  12.53  998.28
Unspecified        251  483.36  464.26  18.45  999.55


## 4. Customer Sentiments across platforms

In [15]:
#Creating a platform sentiment variable that isnt "Unknown"

platform_sentiment = smi[
    smi["Sentiment"] != "Unknown"
]

#Creating a variable to cross reference platform and sentiment
platform_sentiment = pd.crosstab(
    platform_sentiment["Platform"],
    platform_sentiment["Sentiment"]
)

#Print variable
print(platform_sentiment)

Sentiment    Negative  Neutral  Positive  Very Negative  Very Positive
Platform                                                              
Facebook          246      262       267             14              6
Instagram         271      257       287              5             11
Twitter           270      275       244              9             12
Unspecified        77       77       101              6              3


In [26]:
#Creating a variable for platform sentiment that isnt "Unknown" but in percentage (x100)

platform_sentiment_pct = pd.crosstab(
    smi[smi["Sentiment"] != "Unknown"]["Platform"],
    smi[smi["Sentiment"] != "Unknown"]["Sentiment"],
    normalize="index"
) * 100

#Print Percentage
print(platform_sentiment_pct.round(2))

Sentiment    Negative  Neutral  Positive  Very Negative  Very Positive
Platform                                                              
Facebook        30.94    32.96     33.58           1.76           0.75
Instagram       32.61    30.93     34.54           0.60           1.32
Twitter         33.33    33.95     30.12           1.11           1.48
Unspecified     29.17    29.17     38.26           2.27           1.14


## 5. Comparison of Interaction type to sentiment

In [27]:
#Creating a variable that removes the "unknown" category 

interaction_sentiment = smi[
    smi["Sentiment"] != "Unknown"
]

#Making a table variable for interactiontype and sentiment
interaction_sentiment = pd.crosstab(
    interaction_sentiment["InteractionType"],
    interaction_sentiment["Sentiment"]
)

#Print variable
print(interaction_sentiment)

Sentiment        Negative  Neutral  Positive  Very Negative  Very Positive
InteractionType                                                           
Comment               291      300       294             21              8
Like                  289      288       298              3             11
Share                 284      283       307             10             13


In [28]:
#Making the interaction sentiment variable into percentage

interaction_sentiment_pct = pd.crosstab(
    smi[smi["Sentiment"] != "Unknown"]["InteractionType"],
    smi[smi["Sentiment"] != "Unknown"]["Sentiment"],
    normalize="index"
) * 100

#Print Percentage
print(interaction_sentiment_pct.round(2))

Sentiment        Negative  Neutral  Positive  Very Negative  Very Positive
InteractionType                                                           
Comment             31.84    32.82     32.17           2.30           0.88
Like                32.51    32.40     33.52           0.34           1.24
Share               31.66    31.55     34.23           1.11           1.45


## 6. Comparison of amount of transaction vs Payment Type

In [29]:
payment_amount = ct[
    (ct["Amount"] != 0) &
    (ct["Amount"] != -100)
]

print(
    payment_amount.groupby("PaymentMethod")["Amount"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

               count    mean  median    min     max
PaymentMethod                                      
Bank Transfer    713  487.72  491.62  10.18  999.86
Credit Card      725  510.83  508.04  10.03  999.86
Debit Card       737  505.50  494.09  10.14  999.56
PayPal           712  508.45  508.04  10.21  999.55


## 7. Comparison of payment type vs age group

In [30]:
cd_age = cd[["CustomerID", "Age"]].dropna()

age_groups = pd.cut(
    cd_age["Age"],
    bins=[17, 25, 35, 45, 55, 65, 70],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-70"]
)

cd_age = cd_age.copy()
cd_age["AgeGroup"] = age_groups

age_payment = ct.merge(
    cd_age[["CustomerID", "AgeGroup"]],
    on="CustomerID",
    how="inner"
)

payment_age_pct = pd.crosstab(
    age_payment["AgeGroup"],
    age_payment["PaymentMethod"],
    normalize="index"
) * 100

print(payment_age_pct.round(2))

PaymentMethod  Bank Transfer  Credit Card  Debit Card  PayPal
AgeGroup                                                     
18-25                  26.58        22.28       24.81   26.33
26-35                  24.77        25.00       25.23   25.00
36-45                  23.25        26.40       25.82   24.53
46-55                  23.12        27.87       25.89   23.12
56-65                  27.04        21.48       27.04   24.44
66-70                  23.68        27.63       21.49   27.19
